# 03 · Measuring nostalgia in a multilingual corpus

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eabanoz/bocelli-nostalgia/blob/main/notebooks/03_nostalgia_analysis.ipynb)

**Runtime → Change runtime type → T4 GPU.** Full run ≈ 15 minutes on a T4,
over an hour on CPU. Set `SCORE = False` to load the pre-computed scores
from the repo and skip the GPU entirely.

---

## What notebook 02 established

English and Portuguese are **co-dominant** at roughly 35% each; Italian is
8.5%. This is a multilingual corpus by necessity, not by choice.

## Constructs

From Boym's *The Future of Nostalgia* (2001) and adjacent work:

| Construct | Definition |
|---|---|
| **Personal memory** | Autobiographical recall, usually family |
| **Mortality / grief** | Remembrance of the dead; funerals, memorials |
| **Restorative** | The past was better; a wish to return or restore |
| **Reflective** | Bittersweet longing without a wish to restore |
| **Collective** | Shared generational or national memory |
| **Anemoia** | Longing for a time the speaker never lived through |

Restorative vs. reflective matters most: both are nostalgia, but only the
first carries a claim about how things ought to be.

## Method

Two measurements of the same constructs, compared:

1. **A multilingual dictionary** — transparent, fast, and biased toward the
   languages its author reads.
2. **Zero-shot NLI classification** — no word list, works across languages,
   but uncalibrated and unvalidated.

The gap between them, broken down by language, is the methodological finding.

In [ ]:
# --- Clone the repo and set the working directory -------------------------
import sys, os, subprocess
from pathlib import Path

REPO = "bocelli-nostalgia"
if "google.colab" in sys.modules:
    if not Path(REPO).exists():
        subprocess.run(["git", "clone", "-q",
                        "https://github.com/eabanoz/bocelli-nostalgia.git"], check=True)
    ROOT = Path(REPO).resolve()
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
for d in ["data/processed", "outputs/figures"]:
    (ROOT / d).mkdir(parents=True, exist_ok=True)

print("Repo root:", ROOT)

In [ ]:
import pandas as pd, numpy as np, re, time
import matplotlib.pyplot as plt

pd.set_option("display.width", 170); pd.set_option("display.max_colwidth", 88)
plt.rcParams["figure.dpi"] = 110

SCORE = False     # False = load pre-computed scores, no GPU needed

df = pd.read_csv("data/processed/comments_clean.csv")
df["published_at"] = pd.to_datetime(df.published_at, utc=True, errors="coerce")
print(f"Loaded {len(df):,} comments")

KEYS = ["personal_memory", "mortality", "restorative",
        "reflective", "collective", "anemoia"]
KEEP_LANGS = ["en", "pt", "it", "es", "fr", "ru"]

if SCORE:
    import torch
    DEVICE = 0 if torch.cuda.is_available() else -1
    print("GPU:", torch.cuda.get_device_name(0) if DEVICE == 0
          else "NONE — set Runtime > Change runtime type > T4 GPU")

## 1 · Building the analysis set

Three exclusions, each for a different reason.

**Short and low-confidence text.** Language detection is unreliable below
about five tokens, and we need language as a variable.

**Junk language buckets.** `tl`, `nl` and `ro` have mean detection
confidence far below the rest, and inspection shows Portuguese words landing
in Tagalog. They are catch-alls, not languages.

**Pasted text — the important one.** Comment fields contain material that is
not commentary: song lyrics, scripture, spam, study notes. On this video,
~21% of Italian comments are the lyrics of *Con te partirò*, pasted
repeatedly, some still carrying `[Strofa 1]` markers from lyrics sites.

That last exclusion is not housekeeping. The song is about departure, so an
NLI model asked whether the lyrics concern grief and farewell will say yes —
correctly. The text simply is not a comment. Leaving it in produces a large,
plausible, entirely artefactual finding that Italian listeners are far more
nostalgic than everyone else.

In [ ]:
t = df.text_clean.fillna("")

# (a) structural markers copied from lyrics sites
MARKERS = re.compile(
    r"(?i)(?:\[(?:strofa|verse|intro|chorus|ritornello|testo|refrain|"
    r"bridge|outro)[^\]]*\]|^lyrics:|^\s*testo\b)")

# (b) the song's opening line, in Italian and common translations
INCIPIT = re.compile(
    r"(?i)(?:quando sono solo\s*,?\s*(?:e\s+)?sogno all'?orizzonte"
    r"|when i(?:'m| am) alone.{0,30}dream of the horizon"
    r"|cuando estoy solo.{0,30}sue(?:ñ|n)o (?:en|con) el horizonte"
    r"|quando estou s(?:ó|o)zinho.{0,30}sonho (?:com|no) horizonte"
    r"|куандо соно соло)")

# (c) repeated openings — pasted text recurs, genuine comments do not
df["open45"] = (t.str[:45].str.lower()
                .str.replace(r"[^\w\s]", "", regex=True).str.strip())
dup = df.open45.map(df.open45.value_counts()).ge(3) & (df.n_tokens >= 40)

df["is_lyrics"] = t.str.contains(MARKERS) | t.str.contains(INCIPIT) | dup
LONG_CUT = df[df.n_tokens > 0].n_tokens.quantile(0.995)
df["is_pasted"] = df.is_lyrics | ((df.n_tokens > LONG_CUT) & ~df.is_lyrics)

print(f"Lyrics detected : {df.is_lyrics.sum():>5}")
print(f"Total pasted    : {df.is_pasted.sum():>5} "
      f"({100*df.is_pasted.mean():.2f}%)")
print("\nShare excluded as pasted, by language:")
ex = df[df.lang.notna()].groupby("lang").is_pasted.agg(["sum", "count"])
ex["pct"] = (100 * ex["sum"] / ex["count"]).round(1)
print(ex[ex["count"] > 200].sort_values("count", ascending=False).to_string())

In [ ]:
# Always inspect what a filter removes and what it leaves.
print("=== flagged as pasted (sample) ===")
for x in df[df.is_pasted].text_clean.sample(5, random_state=2):
    print("   ", str(x)[:70])
print("\n=== longest NOT flagged (any misses?) ===")
for _, r in df[~df.is_pasted].nlargest(5, "n_tokens").iterrows():
    print(f"   [{int(r.n_tokens)}] {str(r.text_clean)[:70]}")

In [ ]:
JUNK_LANGS = ["tl", "nl", "ro"]

ana = df[df.lang.notna() & ~df.is_textless & ~df.is_pasted &
         (df.n_tokens >= 5) & (df.lang_conf >= 0.55) &
         ~df.lang.isin(JUNK_LANGS)].copy()
ana["lang_grp"] = np.where(ana.lang.isin(KEEP_LANGS), ana.lang, "other")

print(f"Analysis set: {len(ana):,} ({100*len(ana)/len(df):.1f}% of harvest)")
print(ana.lang_grp.value_counts().to_string())
print("\nMedian length by language (watch this — length drives NLI scores):")
print(ana.groupby("lang_grp").n_tokens.agg(["median", "mean", "count"])
        .round(1).sort_values("count", ascending=False).to_string())

## 2 · The dictionary method

Covers English, Italian, Spanish, Portuguese, French, German and Russian.

**Russian is left weak on purpose.** It is ~3.6% of the analysis set and will
show implausibly low rates. Finding that hole is the exercise — every
dictionary has gaps shaped like the languages its author does not read.

In [ ]:
DICT = {
"personal_memory":
  r"(?i)\b(?:my (?:mother|mom|mum|father|dad|grand(?:ma|pa|mother|father)|"
  r"wife|husband|son|daughter|brother|sister)|"
  r"mia (?:madre|mamma|nonna|moglie|sorella)|"
  r"mio (?:padre|papà|nonno|marito|fratello|figlio)|"
  r"mi (?:madre|padre|abuela|abuelo|esposa|esposo|hijo)|"
  r"minha (?:mãe|avó|esposa|irmã|filha)|meu (?:pai|avô|marido|irmão|filho)|"
  r"ma (?:mère|grand-mère|femme)|mon (?:père|grand-père|mari)|"
  r"when i was|quando ero|cuando era|quando eu era|quand j'étais)",
"mortality":
  r"(?i)\b(?:rest in peace|rip\b|r\.i\.p\b|passed away|funeral|"
  r"in memory of|riposa in pace|funerale|è mancat|descanse en paz|falleci|"
  r"descanse em paz|faleceu|saudades|em memória|repose en paix|décédé|"
  r"miss (?:you|him|her)\b|no longer with us)",
"restorative":
  r"(?i)\b(?:real music|music these days|nowadays|today'?s music|bring back|"
  r"la vera musica|musica di oggi|música de verdad|música de verdade|"
  r"música de hoje|não se faz mais|vraie musique|they don'?t make|"
  r"isso sim é música)",
"who_listening":
  r"(?i)\b(?:who(?:'?s| is)? (?:still )?(?:listening|watching)|"
  r"chi (?:ascolta|guarda)|qui[eé]n (?:escucha|sigue)|"
  r"quem (?:est[aá] ouvindo|ouve)|20[12]\d\s*\?)",
}

for k, pat in DICT.items():
    ana[f"dict_{k}"] = ana.text_clean.fillna("").str.contains(
        pat, regex=True, na=False)

dcols = [f"dict_{k}" for k in DICT]
tab = ana.groupby("lang_grp")[dcols].mean().mul(100).round(2)
tab["n"] = ana.lang_grp.value_counts()
display(tab.sort_values("n", ascending=False))
print("Russian at 0.00 across the board — that is the planted hole.")

## 3 · Zero-shot classification

The dictionary asks *does this string appear?* The classifier asks *does this
comment mean this?* — across languages, without a word list.

`mDeBERTa-v3-base-xnli` is trained on multilingual NLI. Each label becomes a
hypothesis and the model scores entailment. English hypotheses against
non-English premises: cross-lingual transfer is what the model is for.

`multi_label=True`, scoring each label **independently**, because a comment
can be a personal memory *and* about mortality at once. Forcing them to
compete measurably degrades the ranking.

In [ ]:
LABELS = {
 "personal_memory": "a personal memory of the commenter's own life or family",
 "mortality":       "someone who has died, grief, or a funeral",
 "restorative":     "music from the past was better than music made today",
 "reflective":      "a bittersweet feeling about time passing and things lost",
 "collective":      "a shared generational, national or cultural memory",
 "anemoia":         "being too young to have lived through the time of this song",
}
LABEL_TEXTS = list(LABELS.values())
KEY_BY_TEXT = {v: k for k, v in LABELS.items()}
zcols = [f"zs_{k}" for k in KEYS]

if SCORE:
    from transformers import pipeline
    zs = pipeline("zero-shot-classification",
                  model="MoritzLaurer/mDeBERTa-v3-base-xnli-"
                        "multilingual-nli-2mil7", device=DEVICE)
    for probe in [
        "My mother used to sing this to me. She passed away in 2019.",
        "Chorei pensando no meu pai, faz 1 ano que ele morreu.",
        "Questa è la vera musica, non come quella di oggi.",
    ]:
        r = zs(probe, candidate_labels=LABEL_TEXTS, multi_label=True)
        top = sorted(zip(r["labels"], r["scores"]), key=lambda x: -x[1])[:2]
        print(f"\n{probe[:60]}")
        for lb, sc in top:
            print(f"    {KEY_BY_TEXT[lb]:16} {sc:.2f}")

In [ ]:
if SCORE:
    texts = ana.text_clean.fillna("vuoto").tolist()
    t0, out, BATCH = time.time(), [], 16
    for i in range(0, len(texts), BATCH):
        res = zs(texts[i:i+BATCH], candidate_labels=LABEL_TEXTS,
                 multi_label=True)
        res = res if isinstance(res, list) else [res]
        for r in res:
            d = dict(zip(r["labels"], r["scores"]))
            out.append({f"zs_{KEY_BY_TEXT[k]}": v for k, v in d.items()})
        if (i // BATCH) % 25 == 0:
            el = time.time() - t0; done = min(i + BATCH, len(texts))
            print(f"  {done:,}/{len(texts):,}  {el:.0f}s, "
                  f"~{el/max(done,1)*(len(texts)-done):.0f}s left")
    work = pd.concat([ana, pd.DataFrame(out, index=ana.index)], axis=1)
    print(f"\nDone in {(time.time()-t0)/60:.1f} min")
else:
    scored = pd.read_csv("data/processed/comments_scored.csv")
    work = ana.merge(scored[["comment_id"] + zcols], on="comment_id",
                     how="inner")
    print(f"Loaded pre-computed scores for {len(work):,} comments")

### Thresholding: rank, do not cut

NLI entailment scores are **not calibrated probabilities**. The model assigns
near-certainty to most inputs — median `zs_mortality` is about 0.98 — so an
absolute cutoff like 0.6 would flag almost everything.

What *is* informative is the ordering. Checked against dictionary hits as a
weak proxy label, the scores reach AUC ≈ 0.91 for personal memory and
mortality: the ranking is sound even though the scale is not. So we flag the
top decile within each construct.

**Say this when reporting:** a top-decile flag guarantees 10% by
construction. You may compare *across languages or years*; you may not say
"10% of comments express mortality."

In [ ]:
from sklearn.metrics import roc_auc_score

print("Does the score rank correctly? (dictionary hits as weak labels)")
for k in ["personal_memory", "mortality", "restorative"]:
    d, z = work[f"dict_{k}"], work[f"zs_{k}"]
    if d.sum() >= 20:
        print(f"  {k:18} AUC = {roc_auc_score(d, z):.3f}")
print("\n~0.90 = ranking is good.  ~0.50 = no signal.")
print("<0.50 = the label means something other than you intended;")
print("        'restorative' fails this test and should not be reported.")

TOP_PCT = 0.10
for k in KEYS:
    work[f"flag_{k}"] = work[f"zs_{k}"] >= work[f"zs_{k}"].quantile(1-TOP_PCT)
fcols = [f"flag_{k}" for k in KEYS]

print("\nFace validity — dictionary hit rate inside vs outside each flag:")
for k in ["mortality", "personal_memory"]:
    i = 100 * work[work[f"flag_{k}"]][f"dict_{k}"].mean()
    o = 100 * work[~work[f"flag_{k}"]][f"dict_{k}"].mean()
    print(f"  {k:18} inside {i:5.1f}%  outside {o:4.1f}%  "
          f"→ {i/max(o,0.01):.0f}x enrichment")

In [ ]:
# What did the classifier catch that the dictionary missed?
for k in ["mortality", "personal_memory"]:
    print(f"\n--- highest {k}, NO dictionary hit ---")
    for _, r in work[~work[f"dict_{k}"]].nlargest(4, f"zs_{k}").iterrows():
        print(f"  [{r[f'zs_{k}']:.3f}] {str(r.text_clean)[:88]}")
print("\nTrue positives in French and Portuguese that the word list missed.")
print("This is the dictionary-vs-classifier argument, shown not asserted.")

## 4 · Dictionary vs. classifier, by language

In [ ]:
comp = []
for lg in KEEP_LANGS:
    s = work[work.lang_grp == lg]
    if len(s) < 100:
        continue
    for k in ["personal_memory", "mortality"]:
        comp.append({"lang": lg, "construct": k,
                     "dict_%": round(100*s[f"dict_{k}"].mean(), 2),
                     "flag_%": round(100*s[f"flag_{k}"].mean(), 2)})
comp = pd.DataFrame(comp)
display(comp.pivot(index="lang", columns="construct",
                   values=["dict_%", "flag_%"]))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for ax, k in zip(axes, ["personal_memory", "mortality"]):
    comp[comp.construct == k].set_index("lang")[["dict_%", "flag_%"]].plot.bar(
        ax=ax, color=["#C44E52", "#4C72B0"])
    ax.set_title(k); ax.set_xlabel(""); ax.tick_params(rotation=0)
    ax.legend(["dictionary", "classifier"], fontsize=7)
plt.tight_layout()
plt.savefig("outputs/figures/dict_vs_classifier.png", bbox_inches="tight")
plt.show()

In [ ]:
lang_tab = work.groupby("lang_grp")[fcols].mean().mul(100).round(1)
lang_tab["n"] = work.lang_grp.value_counts()
lang_tab["median_tokens"] = work.groupby("lang_grp").n_tokens.median()
display(lang_tab[lang_tab.n > 200].sort_values("n", ascending=False))

print("Read this table WITH the median_tokens column. Longer comments score")
print("higher on every construct, so a group that writes at length will")
print("look more nostalgic. Compare groups of similar length, or stratify.")

In [ ]:
# Stratified check: does any language difference survive length control?
work["len_q"] = pd.qcut(work.n_tokens, 4, labels=["Q1", "Q2", "Q3", "Q4"])
for k in ["mortality", "personal_memory"]:
    t = (work[work.lang_grp.isin(["en", "pt", "it", "fr"])]
         .groupby(["len_q", "lang_grp"], observed=True)[f"flag_{k}"]
         .mean().mul(100).unstack().round(1))
    print(f"\n--- flag_{k} (%) by length quartile ---")
    print(t.to_string())
print("\nCompare WITHIN rows, never across them.")

## 5 · Sentiment: the right tool for the wrong question

Expect nostalgic comments to classify as **positive** — they are
affectionate in tone — even when describing a dead parent.

That is not a model failure. Polarity and nostalgia are orthogonal, and
reaching for a sentiment classifier because it is the familiar tool would
have measured something else entirely.

In [ ]:
if SCORE:
    from transformers import pipeline as _p
    sent = _p("text-classification",
              model="cardiffnlp/twitter-xlm-roberta-base-sentiment",
              device=DEVICE, truncation=True, max_length=512)
    preds = sent(work.text_clean.fillna("vuoto").tolist(), batch_size=64)
    work["sentiment"] = [p["label"].lower() for p in preds]
elif "sentiment" in scored.columns:
    work = work.merge(scored[["comment_id", "sentiment"]], on="comment_id",
                      how="left")

if "sentiment" in work.columns:
    print("Overall:")
    print(work.sentiment.value_counts(normalize=True).round(3).to_string())
    print("\nAmong the mortality decile:")
    print(work[work.flag_mortality].sentiment.value_counts(normalize=True)
            .round(3).to_string())
    print("\nGrief reading as 'positive' is the point.")

## 6 · Nostalgia in time

Does nostalgic register intensify as the performance recedes and new
listeners arrive who never experienced 1997?

In [ ]:
yr = work.groupby("comment_year")[fcols].mean().mul(100).round(2)
yr["n"] = work.comment_year.value_counts().sort_index()
yr = yr[yr.n >= 100]
display(yr)

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
yr[["flag_personal_memory", "flag_mortality", "flag_reflective",
    "flag_anemoia"]].plot(ax=ax[0], marker="o")
ax[0].set_title("Construct share by comment year (%)"); ax[0].set_xlabel("")
ax[0].legend(fontsize=8)

ct = pd.crosstab(work.comment_year, work.lang_grp, normalize="index").mul(100)
ct[[c for c in KEEP_LANGS if c in ct.columns]].plot(ax=ax[1], marker="o")
ax[1].set_title("Language mix by year (%)"); ax[1].set_xlabel("")
ax[1].legend(fontsize=8, bbox_to_anchor=(1.02, 1))
plt.tight_layout()
plt.savefig("outputs/figures/nostalgia_over_time.png", bbox_inches="tight")
plt.show()

print("CAUTION: these panels move together. A rise in 'personal memory' may")
print("be a rise in English speakers, not a change in how people relate to")
print("the song. Never report the left panel without the right one.")

In [ ]:
work.drop(columns=["open45"], errors="ignore").to_csv(
    "data/processed/comments_scored.csv", index=False)
lang_tab.to_csv("outputs/constructs_by_language.csv")
comp.to_csv("outputs/dict_vs_classifier.csv", index=False)
print(f"Saved {len(work):,} scored comments")

---

## What to report

1. **Language shares are filter-dependent.** English and Portuguese
   co-dominant at ~35%; the leader moves ten points across reasonable
   specifications.
2. **The dictionary undercounts non-English systematically.** Zero
   Portuguese personal-memory hits before the word list was extended;
   Russian still at zero.
3. **NLI scores are not probabilities.** Rank, do not threshold. AUC ≈ 0.91
   confirms the ordering is sound.
4. **Not all text in a comment field is a comment.** ~21% of Italian
   comments here are pasted song lyrics; leaving them in produces a large
   and entirely artefactual language effect.
5. **Length drives entailment scores.** Report stratified by length.
6. **Sentiment is orthogonal to nostalgia.** Grief classifies as positive.

## Limitations

Everything here is unvalidated: the model has never seen a labelled example
of "reflective nostalgia". These are hypothesis-generating numbers. A proper
study would hand-code a stratified sample and report Cohen's κ for every
construct before publishing any percentage.

`restorative` fails its AUC check (< 0.5) and should not be reported.

Findings describe discourse on one video, not a population. There is no age,
gender or location for any commenter.